# Premier League Shot Analytics

Exploratory analysis of every shot taken in the English Premier League across three seasons.

**Data source:** Understat via `data/pl_goals_fetch_understat.py`  
**File:** `../data/raw/pl_shots_understat.csv`  
**Seasons covered:** 2022/23, 2023/24, 2024/25

**Columns:**
- `x`, `y` — pitch coordinates (0–1 scale; x=0 is own goal, x=1 is opponent's goal)
- `xg` — expected goals value for the shot
- `result` — Goal / SavedShot / MissedShots / BlockedShot / ShotOnPost / OwnGoal
- `situation` — OpenPlay / FromCorner / SetPiece / DirectFreekick / Penalty
- `shot_type` — RightFoot / LeftFoot / Head / OtherBodyPart

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import matplotlib.colors as mcolors
import numpy as np

plt.rcParams['figure.dpi'] = 120
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

df = pd.read_csv('../data/raw/pl_shots_understat.csv')
df['match_date'] = pd.to_datetime(df['match_date'])
goals = df[df['result'] == 'Goal'].copy()

print(f"Total shots: {len(df):,}")
print(f"Total goals: {len(goals):,}")
print(f"Overall conversion rate: {len(goals)/len(df):.1%}")
print(f"Seasons: {sorted(df['season'].unique())}")

## 1. Season overview

In [ ]:
season_summary = (
    df.groupby('season')
    .agg(
        shots=('id', 'count'),
        goals=('result', lambda x: (x == 'Goal').sum()),
        xg_total=('xg', 'sum'),
    )
    .assign(conversion=lambda d: d['goals'] / d['shots'])
    .assign(xg_per_shot=lambda d: d['xg_total'] / d['shots'])
    .sort_index()
)

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
seasons = season_summary.index
colours = ['#1f77b4', '#ff7f0e', '#2ca02c']

axes[0].bar(seasons, season_summary['shots'], color=colours)
axes[0].set_title('Total shots')

axes[1].bar(seasons, season_summary['goals'], color=colours)
axes[1].set_title('Total goals')

axes[2].bar(seasons, season_summary['conversion'], color=colours)
axes[2].set_title('Conversion rate')
axes[2].yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:.0%}'))

for ax in axes:
    ax.tick_params(axis='x', labelrotation=15)

plt.suptitle('Season overview', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

display(season_summary.style.format({'shots': '{:,.0f}', 'goals': '{:,.0f}', 'xg_total': '{:,.1f}', 'conversion': '{:.1%}', 'xg_per_shot': '{:.3f}'}))

## 2. Top scorers (all seasons)

In [ ]:
top_scorers = (
    goals.groupby('player')
    .agg(
        goals=('id', 'count'),
        shots=('id', lambda x: df[df['player'] == x.name].shape[0]),
        xg=('xg', 'sum'),
    )
    .assign(xg_diff=lambda d: d['goals'] - d['xg'])
    .sort_values('goals', ascending=False)
    .head(20)
)

fig, ax = plt.subplots(figsize=(10, 6))
colours = ['#d62728' if v > 0 else '#1f77b4' for v in top_scorers['xg_diff']]
bars = ax.barh(top_scorers.index, top_scorers['goals'], color='#1f77b4', label='Actual goals')
ax.scatter(top_scorers['xg'], range(len(top_scorers)), color='#ff7f0e', zorder=5, s=50, label='xG')
ax.invert_yaxis()
ax.set_xlabel('Goals')
ax.set_title('Top 20 scorers (2022/23 – 2024/25)', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

## 3. xG over/underperformers

Players with the biggest positive or negative difference between actual goals and xG (minimum 20 shots).

In [ ]:
player_stats = (
    df.groupby('player')
    .agg(
        shots=('id', 'count'),
        goals=('result', lambda x: (x == 'Goal').sum()),
        xg=('xg', 'sum'),
    )
    .query('shots >= 20')
    .assign(xg_diff=lambda d: d['goals'] - d['xg'])
)

top_over  = player_stats.nlargest(10, 'xg_diff')
top_under = player_stats.nsmallest(10, 'xg_diff')
combined  = pd.concat([top_over, top_under]).sort_values('xg_diff')

fig, ax = plt.subplots(figsize=(10, 6))
colours = ['#d62728' if v > 0 else '#1f77b4' for v in combined['xg_diff']]
ax.barh(combined.index, combined['xg_diff'], color=colours)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Goals minus xG')
ax.set_title('xG over/underperformers (min. 20 shots)', fontweight='bold')
ax.annotate('Overperforming →', xy=(combined['xg_diff'].max() * 0.6, 1), fontsize=8, color='#d62728')
ax.annotate('← Underperforming', xy=(combined['xg_diff'].min() * 0.6, 1), fontsize=8, color='#1f77b4', ha='right')
plt.tight_layout()
plt.show()

## 4. Goals by situation and shot type

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Situation breakdown — goals and conversion rate
sit = (
    df.groupby('situation')
    .agg(shots=('id', 'count'), goals=('result', lambda x: (x == 'Goal').sum()))
    .assign(conversion=lambda d: d['goals'] / d['shots'])
    .sort_values('goals', ascending=True)
)
ax = axes[0]
ax.barh(sit.index, sit['goals'], color='#1f77b4')
for i, (idx, row) in enumerate(sit.iterrows()):
    ax.text(row['goals'] + 5, i, f"{row['conversion']:.0%}", va='center', fontsize=9, color='grey')
ax.set_title('Goals by situation\n(conversion rate shown)', fontweight='bold')
ax.set_xlabel('Goals')

# Shot type breakdown
stype = (
    df.groupby('shot_type')
    .agg(shots=('id', 'count'), goals=('result', lambda x: (x == 'Goal').sum()))
    .assign(conversion=lambda d: d['goals'] / d['shots'])
    .sort_values('goals', ascending=True)
)
ax = axes[1]
ax.barh(stype.index, stype['goals'], color='#ff7f0e')
for i, (idx, row) in enumerate(stype.iterrows()):
    ax.text(row['goals'] + 5, i, f"{row['conversion']:.0%}", va='center', fontsize=9, color='grey')
ax.set_title('Goals by shot type\n(conversion rate shown)', fontweight='bold')
ax.set_xlabel('Goals')

plt.tight_layout()
plt.show()

## 5. When are goals scored? (goals by minute)

In [ ]:
# Cap at minute 95 to group stoppage time together
goals_plot = goals.copy()
goals_plot['minute_capped'] = goals_plot['minute'].clip(upper=95)

n_matches = df['match_id'].nunique()

fig, ax = plt.subplots(figsize=(12, 4))
ax.hist(
    goals_plot['minute_capped'],
    bins=range(1, 97),
    weights=[1 / n_matches] * len(goals_plot),
    color='#2ca02c', edgecolor='white', linewidth=0.3,
)
ax.axvline(45, color='grey', linestyle='--', linewidth=0.8, label='Half time')
ax.set_xlabel('Minute')
ax.set_ylabel('Goals per match')
ax.set_title('Goals per match by minute of the game (all seasons)', fontweight='bold')
ax.set_xticks([1, 15, 30, 45, 60, 75, 90, 95])
ax.set_xticklabels(['1', '15', '30', '45', '60', '75', '90', '90+'])
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:.1%}'))
ax.yaxis.grid(True, linestyle='--', linewidth=0.5, color='grey', alpha=0.7)
ax.set_axisbelow(True)
ax.legend()
plt.tight_layout()
plt.show()

## 5b. Goals by minute, split by half-time score state

For each match we derive the half-time score from goals scored in minutes 1–45, then ask: does the distribution of second-half goals look different depending on whether the home side is winning, drawing, or losing at the break?

In [ ]:
# --- Derive half-time score per match -----------------------------------
first_half = df[(df['result'].isin(['Goal', 'OwnGoal'])) & (df['minute'] <= 45)].copy()

def ht_team_goals(row):
    """Return (home_goals, away_goals) contribution of a single goal row."""
    if row['result'] == 'Goal':
        return (1, 0) if row['side'] == 'h' else (0, 1)
    else:  # OwnGoal — counts for the other side
        return (0, 1) if row['side'] == 'h' else (1, 0)

contributions = first_half.apply(ht_team_goals, axis=1, result_type='expand')
contributions.columns = ['home', 'away']
first_half = first_half.join(contributions)

ht_scores = (
    first_half.groupby('match_id')[['home', 'away']]
    .sum()
    .rename(columns={'home': 'ht_home', 'away': 'ht_away'})
)

ht_scores['ht_gd']    = (ht_scores['ht_home'] - ht_scores['ht_away']).abs()
ht_scores['ht_state'] = ht_scores['ht_gd'].apply(
    lambda gd: 'Close (GD 0–1)' if gd <= 1 else 'Comfortable (GD 2+)'
)

print("Matches per HT state:")
print(ht_scores['ht_state'].value_counts().to_string())

# --- Plot: second-half goals per match, 3-minute bins, two HT states ----
second_half_goals = (
    goals[goals['minute'] > 45]
    .merge(ht_scores[['ht_state']], on='match_id')
    .copy()
)
second_half_goals['minute_capped'] = second_half_goals['minute'].clip(upper=95)

states  = ['Close (GD 0–1)', 'Comfortable (GD 2+)']
colours = ['#1f77b4', '#d62728']
bins    = range(46, 98, 3)  # 3-minute intervals

fig, ax = plt.subplots(figsize=(12, 5))

for state, colour in zip(states, colours):
    subset = second_half_goals[second_half_goals['ht_state'] == state]
    n      = ht_scores[ht_scores['ht_state'] == state].shape[0]
    ax.hist(
        subset['minute_capped'],
        bins=bins,
        weights=[1 / n] * len(subset),
        histtype='step',
        linewidth=2,
        color=colour,
        label=f'{state} (n={n:,} matches)',
    )

ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:.1%}'))
ax.yaxis.grid(True, linestyle='--', linewidth=0.5, color='grey', alpha=0.7)
ax.set_axisbelow(True)
ax.set_xlabel('Minute')
ax.set_ylabel('Goals per match')
ax.set_title('Second-half goals per match (3-min bins) — close vs comfortable HT scorelines', fontweight='bold')
ax.set_xticks(list(range(46, 93, 6)) + [95])
ax.set_xticklabels([str(m) for m in range(46, 93, 6)] + ['90+'])
ax.legend()
plt.tight_layout()
plt.show()

## 6. Shot locations on the pitch

Understat coordinates use a 0–1 scale where x=1 is the attacking goal. To plot on a standard pitch (105m × 68m) we rescale.

In [ ]:
def draw_half_pitch(ax, lw=1.2):
    """Draw the attacking half of a pitch (origin = centre spot)."""
    pitch_colour = '#2d5a27'
    line_colour  = 'white'
    goal_lw      = lw * 1.5

    ax.set_facecolor(pitch_colour)

    # Pitch outline (attacking half: x 52.5–105, full width 0–68)
    ax.add_patch(patches.Rectangle((52.5, 0), 52.5, 68, fill=False, edgecolor=line_colour, lw=lw, zorder=3))

    # Halfway line
    ax.plot([52.5, 52.5], [0, 68], color=line_colour, lw=lw, zorder=3)

    # Penalty area (16.5m from goal line, 40.32m wide centred)
    ax.add_patch(patches.Rectangle((88.5, 13.84), 16.5, 40.32, fill=False, edgecolor=line_colour, lw=lw, zorder=3))

    # 6-yard box
    ax.add_patch(patches.Rectangle((99, 24.84), 6, 18.32, fill=False, edgecolor=line_colour, lw=lw, zorder=3))

    # Goal
    ax.add_patch(patches.Rectangle((105, 30.34), 2, 7.32, fill=False, edgecolor='gold', lw=goal_lw, zorder=3))

    # Penalty spot
    ax.plot(93.5, 34, 'o', color=line_colour, ms=2, zorder=3)

    # Penalty arc
    arc = patches.Arc((93.5, 34), 18.3, 18.3, angle=0, theta1=128, theta2=232, color=line_colour, lw=lw, zorder=3)
    ax.add_patch(arc)

    ax.set_xlim(50, 107)
    ax.set_ylim(-1, 69)
    ax.set_aspect('equal')
    ax.axis('off')


# Convert Understat coords to metres
# Understat x: 0=own goal end, 1=opp goal end  → multiply by 105
# Understat y: 0=left touchline, 1=right touchline → multiply by 68
df['x_m'] = df['x'] * 105
df['y_m'] = df['y'] * 68
goals['x_m'] = goals['x'] * 105
goals['y_m'] = goals['y'] * 68

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# All shots — scatter first, then pitch lines on top
axes[0].set_facecolor('#2d5a27')
axes[0].scatter(
    df['x_m'], df['y_m'],
    c=df['xg'], cmap='YlOrRd', s=4, alpha=0.3, vmin=0, vmax=0.5, zorder=1,
)
draw_half_pitch(axes[0], lw=2)
axes[0].set_title('All shots (coloured by xG)', color='white', fontweight='bold', pad=8)

# Goals only — scatter first, then pitch lines on top
axes[1].set_facecolor('#2d5a27')
axes[1].scatter(
    goals['x_m'], goals['y_m'],
    c=goals['xg'], cmap='YlOrRd', s=10, alpha=0.5, vmin=0, vmax=0.5, zorder=1,
)
draw_half_pitch(axes[1], lw=2)
axes[1].set_title('Goals only (coloured by xG)', color='white', fontweight='bold', pad=8)

fig.patch.set_facecolor('#2d5a27')
sm = plt.cm.ScalarMappable(cmap='YlOrRd', norm=mcolors.Normalize(vmin=0, vmax=0.5))
cbar = fig.colorbar(sm, ax=axes, shrink=0.6, pad=0.02)
cbar.set_label('xG', color='white')
cbar.ax.yaxis.set_tick_params(color='white')
plt.setp(cbar.ax.yaxis.get_ticklabels(), color='white')

plt.suptitle('Shot locations — EPL 2022/23 to 2024/25', color='white', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 7. Shot heatmap by pitch zone

2D histogram showing where shots are most frequently taken from.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

titles = ['All shots', 'Goals only']
datasets = [df, goals]

for ax, title, data in zip(axes, titles, datasets):
    ax.set_facecolor('#2d5a27')
    h, xedges, yedges = np.histogram2d(
        data['x_m'], data['y_m'],
        bins=[np.linspace(52.5, 105, 30), np.linspace(0, 68, 20)]
    )
    ax.pcolormesh(
        xedges, yedges, h.T,
        cmap='hot', alpha=0.65,
        norm=mcolors.PowerNorm(gamma=0.4),
        zorder=1,
    )
    # Draw pitch lines after the heatmap so they sit on top
    draw_half_pitch(ax, lw=2)
    ax.set_title(title, color='white', fontweight='bold', pad=8)

fig.patch.set_facecolor('#2d5a27')
plt.suptitle('Shot density heatmap — EPL 2022/23 to 2024/25', color='white', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()